## Setting up agents

In [1]:
# LLM
from langchain_community.chat_models import ChatOpenAI

# Loader
from langchain_community.document_loaders import TextLoader

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector store
from langchain_community.vectorstores import Chroma

# Tools & Agent
from langchain_core.tools import Tool
from langchain.agents import create_react_agent, AgentExecutor


In [2]:
llm = ChatOpenAI(
    openai_api_base="http://localhost:1234/v1",
    openai_api_key="lm-studio",
    model="qwen2.5-0.5b-instruct",
    temperature=0
)

d:\Shri\Study\AIML_Engineer\venv\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


## Creating the tools 

In [3]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/health_data.txt")
documents = loader.load()

In [4]:
for doc in documents:
    print(doc.page_content)

DOCUMENT: HealthCare Support Guidelines v1.2

SECTION 1: GENERAL INFORMATION
HealthCare Support is a program designed to assist patients with medication access, eligibility, and adverse event reporting. The system ensures compliance with regulatory requirements.

SECTION 2: ELIGIBILITY RULES
Patients are eligible for support if they meet the following criteria:
- Must be 18 years or older
- Must be a resident of the United States
- Must have a valid prescription from a licensed physician

Exceptions:
Patients under 18 may be eligible under special pediatric programs if approved by a medical reviewer.

SECTION 3: ADVERSE EVENT (AE) REPORTING
An adverse event is defined as any undesirable experience associated with the use of a medical product.

Examples include:
- Headache
- Nausea
- Dizziness
- Severe allergic reactions

All adverse events must be reported within 24 hours of awareness.

SECTION 4: PRODUCT COMPLAINT (PC)
A product complaint refers to issues related to the product qualit

## Chunking

In [31]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)

docs = splitter.split_documents(documents)

In [32]:

docs

[Document(page_content='DOCUMENT: HealthCare Support Guidelines v1.2\n\nSECTION 1: GENERAL INFORMATION\nHealthCare Support is a program designed to assist patients with medication access, eligibility, and adverse event reporting. The system ensures compliance with regulatory requirements.', metadata={'source': 'data/health_data.txt'}),
 Document(page_content='SECTION 2: ELIGIBILITY RULES\nPatients are eligible for support if they meet the following criteria:\n- Must be 18 years or older\n- Must be a resident of the United States\n- Must have a valid prescription from a licensed physician\n\nExceptions:\nPatients under 18 may be eligible under special pediatric programs if approved by a medical reviewer.', metadata={'source': 'data/health_data.txt'}),
 Document(page_content='SECTION 3: ADVERSE EVENT (AE) REPORTING\nAn adverse event is defined as any undesirable experience associated with the use of a medical product.\n\nExamples include:\n- Headache\n- Nausea\n- Dizziness\n- Severe alle

In [33]:
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print()

--- Chunk 1 ---
DOCUMENT: HealthCare Support Guidelines v1.2

SECTION 1: GENERAL INFORMATION
HealthCare Support is a program designed to assist patients with medication access, eligibility, and adverse event reporting. The system ensures compliance with regulatory requirements.

--- Chunk 2 ---
SECTION 2: ELIGIBILITY RULES
Patients are eligible for support if they meet the following criteria:
- Must be 18 years or older
- Must be a resident of the United States
- Must have a valid prescription from a licensed physician

Exceptions:
Patients under 18 may be eligible under special pediatric programs if approved by a medical reviewer.

--- Chunk 3 ---
SECTION 3: ADVERSE EVENT (AE) REPORTING
An adverse event is defined as any undesirable experience associated with the use of a medical product.

Examples include:
- Headache
- Nausea
- Dizziness
- Severe allergic reactions

All adverse events must be reported within 24 hours of awareness.

--- Chunk 4 ---
All adverse events must be reported 

In [34]:
chunk_overlap = 100  # or whatever value you set
for i in range(len(docs) - 1):
    print(f"Chunk {i+1} end: {docs[i].page_content[-chunk_overlap:]}")
    print(f"Chunk {i+2} start: {docs[i+1].page_content[:chunk_overlap]}")
    print("-" * 40)

Chunk 1 end: ligibility, and adverse event reporting. The system ensures compliance with regulatory requirements.
Chunk 2 start: SECTION 2: ELIGIBILITY RULES
Patients are eligible for support if they meet the following criteria:

----------------------------------------
Chunk 2 end: atients under 18 may be eligible under special pediatric programs if approved by a medical reviewer.
Chunk 3 start: SECTION 3: ADVERSE EVENT (AE) REPORTING
An adverse event is defined as any undesirable experience as
----------------------------------------
Chunk 3 end: iness
- Severe allergic reactions

All adverse events must be reported within 24 hours of awareness.
Chunk 4 start: All adverse events must be reported within 24 hours of awareness.

SECTION 4: PRODUCT COMPLAINT (PC)
----------------------------------------
Chunk 4 end: g
- Missing tablets
- Incorrect labeling

Product complaints do NOT include health-related symptoms.
Chunk 5 start: Product complaints do NOT include health-related symptoms.

## Embedding

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

d:\Shri\Study\AIML_Engineer\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Shri\Study\AIML_Engineer\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to a

In [14]:
from langchain.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    docs,
    embedding=embeddings,
    persist_directory="./data/chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [39]:
retriever.get_relevant_documents("can you tell me more on savings card")

[Document(page_content='SECTION 5: SAVINGS CARD SUPPORT\nPatients may request savings cards for eligible medications.\n\nProcess:\n1. User requests savings card\n2. System asks for phone number\n3. User confirms phone number\n4. Card link is sent via SMS\n\nSECTION 6: VERBATIM RESPONSE RULE\nIf a user asks a question that exactly matches a stored FAQ, the system MUST return the exact answer without modification.', metadata={'source': 'data/health_data.txt'}),
 Document(page_content='All adverse events must be reported within 24 hours of awareness.\n\nSECTION 4: PRODUCT COMPLAINT (PC)\nA product complaint refers to issues related to the product quality, packaging, or delivery.\n\nExamples:\n- Damaged packaging\n- Missing tablets\n- Incorrect labeling\n\nProduct complaints do NOT include health-related symptoms.\n\nSECTION 5: SAVINGS CARD SUPPORT\nPatients may request savings cards for eligible medications.', metadata={'source': 'data/health_data.txt'}),
 Document(page_content='DOCUMENT:

## Create Tools for Agent

In [40]:
from langchain.tools import Tool

def retrieve_docs(query):
    results = retriever.get_relevant_documents(query)
    return "\n\n".join([doc.page_content for doc in results])

In [41]:
retrieve_tool = Tool(
    name="retrieve_docs",
    func=retrieve_docs,
    description="Use this to fetch relevant policy documents"
)

## Add Verbatim Tool

In [50]:
FAQ = {
    "What is an adverse event?":
    "An adverse event is defined as any undesirable experience associated with the use of a medical product."
}

def verbatim_check(query):
    normalized_query = query.strip().lower()
    for key in FAQ:
        if key.strip().lower() == normalized_query:
            return FAQ[key]
    return "NO_MATCH"

verbatim_tool = Tool(
    name="verbatim_check",
    func=verbatim_check,
    description="Check if question has exact predefined answer"
)

## Create Agent

In [51]:
from langchain.agents import initialize_agent, AgentType


tools = [retrieve_tool, verbatim_tool]

agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

## Run it

In [55]:
query = "who is obama ?"
response = agent.run(query)

print(response)



> Entering new AgentExecutor chain...
I understand that you want me to provide information about Barack Obama. To do this, I will use the "retrieve_docs" tool to fetch relevant policy documents related to Obama and then analyze them to determine his identity.
Action: retrieve_docs
Action Input: {"query": "Barack Obama"}
Observation: SECTION 5: SAVINGS CARD SUPPORT
Patients may request savings cards for eligible medications.

Process:
1. User requests savings card
2. System asks for phone number
3. User confirms phone number
4. Card link is sent via SMS

SECTION 6: VERBATIM RESPONSE RULE
If a user asks a question that exactly matches a stored FAQ, the system MUST return the exact answer without modification.

DOCUMENT: HealthCare Support Guidelines v1.2

SECTION 1: GENERAL INFORMATION
HealthCare Support is a program designed to assist patients with medication access, eligibility, and adverse event reporting. The system ensures compliance with regulatory requirements.

SECTION 2: ELIGI

BadRequestError: Error code: 400 - {'error': 'The number of tokens to keep from the initial prompt is greater than the context length (n_keep: 4336>= n_ctx: 4096). Try to load the model with a larger context length, or provide a shorter input.'}

In [54]:
response

'Adverse events are unexpected and undesirable outcomes that occur during a medical procedure or treatment. They can be caused by various factors, such as medication side effects, complications from surgery, infections, or other health issues.'